# Phase 4 · Court Judgment Prediction — Knowledge distillation: Architecture-4 teacher → Architecture-2 encoder student

| | |
|---|---|
| **Codabench** | Macro-F1 **0.5937** · Precision 0.6 · Recall 0.5656 (rank 7 of 11 in Phase 4) |
| **Own dev / validation** | dev macro-F1: teacher 0.6304 (AUC 0.6568), student 0.6395 (AUC 0.6711, best epoch 3) |
| **Backbone** | Architecture-2 encoder-decoder (150M) |

**The task, in simple words:** predict the **outcome** of a case (appeal accepted = 1 / rejected = 0) from its text.

**How this notebook solves it:**
1. Keep only the **last 1,536 tokens** of each judgment (the decision is written near the end) as 3 chunks of 512.
2. The Phase-1 model turns every chunk into one vector.
3. A BiLSTM reads the 3 chunk vectors in order and attention decides which chunk matters most → P(accepted).
4. A threshold chosen on dev turns the probability into 0 / 1.

**What is special in this run:** **Knowledge distillation** — a larger / stronger *teacher* is trained first; a smaller *student* then learns from the gold labels **and** from the teacher's soft predictions, to keep most of the quality at a lower cost.

*Exact setting:* head + token embeddings of each Phase-1 model (partial fine-tuning); transformer blocks frozen.

**To run it:** Attach the tokenizer notebook's output and the Phase-1 checkpoints (teacher and student) as Kaggle inputs, add the Hugging Face token as the Kaggle secret **`HF`**, pick a GPU (T4), then *Run All*. Paths are in the configuration cell; if a checkpoint is not found there, the notebook searches `/kaggle/input`. Test labels are never used.

Every step below starts with a short explanation: first in simple words, then how it works exactly.

### CELL 1 · Imports + reproducibility + device

**In simple words:** Load the Python libraries, fix the random seed (so a re-run gives the same result) and pick the GPU.

**How it works:** Seeds for `random`, NumPy and PyTorch; `DEVICE` is the GPU when one is attached.

In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


### CELL 2 · Configuration

**In simple words:** Every setting of this run in one place: input paths, sizes, learning rates, epochs. To change the experiment, change only this cell.

**How it works:** The comment block at the top summarises the whole method. Mixed precision: float16 + GradScaler on T4 / P100, bfloat16 on A100. `DEBUG_MODE = True` (where present) runs a few-minute check on a small subset.

In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# Phase 4 (Court Judgment Prediction): KNOWLEDGE DISTILLATION
#
#   TEACHER = Phase-1 Architecture-4 hybrid prefix-LM (205.5M, the LARGEST Phase-1 model),
#             fine-tuned on CJPE in this notebook (STEP 1, CELL 13)
#   STUDENT = Phase-1 Architecture-2 encoder (~68M weights used, the SMALLEST and fastest),
#             trained with the teacher's probabilities (STEP 3, CELL 15) -> submitted
#
#   Manual 10.2.2: "use a larger or stronger teacher model and transfer its prediction
#   knowledge to a smaller student model". Both start from Phase-1 models; CJPE data only.
#
#   both read the same input: the LAST 1536 tokens of a case = 3 chunks of 512 tokens
#   both use the same head: mix of ALL layers -> token attention -> BiLSTM over the
#        chunks -> 0 / 1 (the last layer alone hides the words that decide a case)
#   both are fine-tuned the same way: the head + the TOKEN EMBEDDINGS of the Phase-1
#        model (partial fine-tuning); the transformer blocks stay frozen
#   student loss = (1 - KD_ALPHA) * cross-entropy with the gold label
#                  + KD_ALPHA * T^2 * KL(teacher probabilities || student probabilities)
#   both classifiers START as a logistic regression on the frozen features (CELL 11)
#        + a deep path that starts at 0; epoch 0 = this start is kept if no epoch beats it
#
# v3. v2 on the real data: teacher 0.602 AUC at the start -> 0.602 / 0.607 / 0.600 after
#     training the head / top 8 blocks / all 16 blocks (65 min): fine-tuning the BLOCKS did
#     not help. The outcome is carried by WHICH WORDS appear (word counts: dev AUC 0.71) and
#     the token embeddings are the best Phase-1 layer (probe 0.606). v3 trains the token
#     embeddings instead: every token learns its own contribution (fast: the blocks run
#     without gradients). The linear path now reads chunk-standardized features (stable
#     when the features change during training).
#
# Trained on TRAIN only; dev picks the best epochs and the decision threshold.
# ============================================================

RUN_NAME = "p4_kd_arch4_to_arch2_v3"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 checkpoints. If a path is wrong, CELL 9 searches /kaggle/input for it.
TEACHER_CHECKPOINT = Path(       # Architecture-4 (HybridLM) best.pt
    "/kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-4/model4_checkpoints/best.pt"
)
STUDENT_CHECKPOINT = Path(       # Architecture-2 (encoder-decoder) best_encoder_decoder.pt
    "/kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-2/architecture_2_encoder_decoder/best_encoder_decoder.pt"
)

# Optional: a teacher that is ALREADY trained = teacher_trainable.pt saved by THIS notebook (v3)
# in an earlier run. It skips STEP 1 (~35 min). None = train the teacher here.
TEACHER_TRAINED_PATH = None

# Phase-4 data: None = download from Hugging Face (Kaggle secret "HF"),
# or the folder that contains Train/ and Test/, e.g. Path("/kaggle/input/phase-4-cjpe/Phase_4_CJPE")
LOCAL_DATA_DIR = None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

OUTPUT_DIR = Path(f"/kaggle/working/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Which part of the (long) document the models read
# ------------------------------------------------------------

DOC_PART = "end"             # "end" = the last DOC_TOKENS tokens, "start", or "start_end"
DOC_TOKENS = 1536            # tokens per document
CHUNK_LEN = 512              # every chunk goes through the model separately


# ------------------------------------------------------------
# Knowledge distillation (Hinton, Vinyals & Dean, 2015)
# ------------------------------------------------------------

KD_ALPHA = 0.5               # weight of the teacher term (0 = no distillation)
KD_TEMPERATURE = 1.0         # 2 classes and soft teacher probabilities -> T = 1
TEACHER_SCORED_DOCS = None   # train cases the teacher scores (= the student's train cases).
                             # None = all (~55 min with Architecture-4); 20000 saves ~25 min


# ------------------------------------------------------------
# Head (same for teacher and student)
# ------------------------------------------------------------

POOL_SKIP_TOKENS = 4         # the first tokens of a chunk are not pooled (they behave abnormally in LMs)
LSTM_HIDDEN = 256
HEAD_DROPOUT = 0.1
LR_HEAD = 1e-3

CLASS_BALANCING = True       # loss weights = inverse class frequency (macro-F1 counts both classes)


# ------------------------------------------------------------
# Fine-tuning (same for teacher and student): head + TOKEN EMBEDDINGS of the Phase-1 model
# ------------------------------------------------------------

TRAIN_EMBEDDINGS = True      # False = head only (the Phase-1 model stays completely frozen)
LR_EMBEDDINGS = 1e-4         # token embeddings (AdamW, no weight decay); 3e-4 made the loss jump at the start

BACKBONE_DROPOUT = 0.0       # Phase-1 dropout inside both models is switched off
PREFIX_LENGTH = None         # Architecture-4 bidirectional prefix: None = the Phase-1 prefix_length


# ------------------------------------------------------------
# Training (TRAIN data only)
#   Architecture-4 needs ~3x the compute of the Architecture-2 encoder per case
#   epoch 0 = the start (logistic regression), then epochs of head + token-embedding training
# ------------------------------------------------------------

TEACHER_EPOCHS = 3           # epoch = EPOCH_DOCS cases, then dev is checked
TEACHER_EPOCH_DOCS = 8000
TEACHER_BATCH_DOCS = 8       # documents per forward pass (the blocks run without gradients). If CUDA out-of-memory: 4
TEACHER_GRAD_ACCUM = 2       # -> 16 documents per weight update

STUDENT_EPOCHS = 4
STUDENT_EPOCH_DOCS = 12000
STUDENT_BATCH_DOCS = 8       # If CUDA out-of-memory: 4
STUDENT_GRAD_ACCUM = 2       # -> 16 documents per weight update

PATIENCE = 2                 # stop if dev macro-F1 does not improve for 2 epochs
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0

EVAL_CHUNKS = 24             # chunks per forward pass when predicting

# Before each training (CELL 11): a linear probe on every layer of the frozen Phase-1 model
# (PROBE_DOCS train cases, scored on dev) + token statistics from STATS_DOCS train cases.
STATS_DOCS = 1000
PROBE_DOCS = 5000


# ------------------------------------------------------------
# Debug mode (quick run on a few documents)
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 64              # per split

if DEBUG_MODE:
    TEACHER_EPOCHS = STUDENT_EPOCHS = 2
    TEACHER_EPOCH_DOCS = STUDENT_EPOCH_DOCS = 64
    PROBE_DOCS = 64
    STATS_DOCS = 64
    SIGNAL_TRAIN_DOCS = 64


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


assert DOC_PART in ("end", "start", "start_end")
assert DOC_TOKENS % CHUNK_LEN == 0

MAX_CHUNKS = DOC_TOKENS // CHUNK_LEN

print("Run       :", RUN_NAME)
print("Reads     :", f"{DOC_PART} of each document, {DOC_TOKENS} tokens = {MAX_CHUNKS} chunks x {CHUNK_LEN}")
print("KD        :", f"teacher Architecture-4 -> student Architecture-2 encoder | alpha {KD_ALPHA}, T {KD_TEMPERATURE}")
print("Output    :", OUTPUT_DIR)
print("AMP dtype :", AMP_DTYPE if USE_AMP else "disabled")

Run       : p4_kd_arch4_to_arch2_v3
Reads     : end of each document, 1536 tokens = 3 chunks x 512
KD        : teacher Architecture-4 -> student Architecture-2 encoder | alpha 0.5, T 1.0
Output    : /kaggle/working/p4_kd_arch4_to_arch2_v3
AMP dtype : torch.float16


### CELL 3 · Load the Phase-1 tokenizer + fast encoding

**In simple words:** Load our own Phase-1 BPE tokenizer — the same one the Phase-1 model was trained with. Any other tokenizer would give the model token ids it has never seen.

**How it works:** `tokenizer.py` from the tokenizer notebook. Long documents: only the needed part is tokenized, and every pre-token piece is BPE-encoded once and cached, which makes tokenizing thousands of judgments fast.

In [3]:
# ============================================================
# CELL 3 — LOAD THE PHASE-1 TOKENIZER + FAST ENCODING
#
# Documents are long, so only the part we need is tokenized
# (e.g. the last ~DOC_TOKENS tokens), and every pre-token piece
# is BPE-encoded only once (cache).
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


piece_cache = {}      # pre-token piece -> token ids


def encode_fast(text):
    """Same ids as tokenizer.encode(text), but every piece is encoded only once."""

    ids = []

    for piece in tok_module.PRETOKEN_PATTERN.findall(tok_module.normalize_text(text)):

        if piece not in piece_cache:
            piece_cache[piece] = tokenizer.encode(piece)

        ids.extend(piece_cache[piece])

    return ids


def head_tokens(text, n):
    """First n tokens of text (tokenizes only the beginning, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[:n]

        part = text[:chars]
        part = part[:max(part.rfind(" "), 1)]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[:n]

        chars *= 2


def tail_tokens(text, n):
    """Last n tokens of text (tokenizes only the end, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[-n:]

        part = text[-chars:]
        part = part[max(part.find(" "), 0):]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[-n:]

        chars *= 2


demo = "The appeal is allowed. The order of the High Court dated 12.03.2010 is set aside."
assert encode_fast(demo) == tokenizer.encode(demo)
print("Demo:", [tokenizer.id_to_token[i] for i in encode_fast(demo)][:12])

Vocabulary size: 32004 | PAD / UNK: 0 1
Demo: ['The', ' ', 'appeal', ' ', 'is', ' ', 'allow', 'ed', '.', ' ', 'The', ' ']


### CELL 4 · Phase-4 data files (download needs the Kaggle secret "HF")

**In simple words:** Download the official data of this phase from Hugging Face.

**How it works:** The dataset is gated: the token comes from the Kaggle secret `HF` (never written in the notebook). Files are cached, so a re-run does not download again.

In [4]:
# ============================================================
# CELL 4 — PHASE-4 DATA FILES  (download needs the Kaggle secret "HF")
# ============================================================

PHASE4_FILES = {
    "train": "Phase_4_CJPE/Train/train.json",
    "dev":   "Phase_4_CJPE/Train/dev.json",
    "test":  "Phase_4_CJPE/Test/test.json",
}

DATA_FILES = {}

if LOCAL_DATA_DIR is not None:

    for name, filename in PHASE4_FILES.items():
        DATA_FILES[name] = Path(LOCAL_DATA_DIR) / filename.replace("Phase_4_CJPE/", "")

else:

    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    for name, filename in PHASE4_FILES.items():

        DATA_FILES[name] = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )

for name, path in DATA_FILES.items():
    print(f"{name:6s} -> {path}  ({os.path.getsize(path) / 1e6:.0f} MB)")

train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json  (737 MB)
dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json  (45 MB)
test   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json  (35 MB)


### CELL 5 · Load cases + statistics

**In simple words:** Read the train / dev / test cases: the text and the outcome label (none for test).

In [5]:
# ============================================================
# CELL 5 — LOAD CASES + STATISTICS
#
# Every case: {"id", "text", "label" (0 / 1, or None for test)}
# ============================================================

LABEL_KEYS = ["label", "labels", "judgment", "decision", "y"]


def read_json(path):
    """A JSON file, or a JSON-lines file (one case per line)."""

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return [json.loads(line) for line in content.splitlines() if line.strip()]


def load_cases(path, with_labels=True):

    raw = read_json(path)

    # accepted layouts: [case, ...]  |  {"data": [case, ...]}  |  {case_id: case}
    if isinstance(raw, dict):

        list_key = next((k for k in ("data", "cases", "documents") if isinstance(raw.get(k), list)), None)

        if list_key is not None:
            raw = raw[list_key]
        else:
            raw = [dict(value, id=key) if isinstance(value, dict) else {"id": key, "text": value}
                   for key, value in raw.items()]

    cases = []

    for i, item in enumerate(raw):

        label_key = next((k for k in LABEL_KEYS if k in item), None) if with_labels else None
        label = item[label_key] if label_key is not None else None

        if isinstance(label, list):                 # e.g. [1]
            label = label[0]

        cases.append({
            "id": str(item.get("id", i)),
            "text": str(item.get("text", "")),
            "label": None if label is None else int(label),
        })

    if DEBUG_MODE:
        cases = cases[:DEBUG_DOCS]

    return cases


train_cases = load_cases(DATA_FILES["train"])
dev_cases = load_cases(DATA_FILES["dev"])
test_cases = load_cases(DATA_FILES["test"], with_labels=False)     # test labels are private: never read

for name, cases in [("train", train_cases), ("dev", dev_cases)]:
    assert all(c["label"] in (0, 1) for c in cases), f"{name}: labels must be 0 or 1"

ids = [c["id"] for c in test_cases]
assert len(ids) == len(set(ids)), "test ids are not unique"


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(f"{'split':6s} {'cases':>7s} {'label 0':>8s} {'label 1':>8s} {'chars (median / 90% / max)':>30s}")

for name, cases in [("train", train_cases), ("dev", dev_cases), ("test", test_cases)]:

    labels = Counter(c["label"] for c in cases)
    chars = np.array([len(c["text"]) for c in cases])

    print(f"{name:6s} {len(cases):7d} {labels.get(0, 0):8d} {labels.get(1, 0):8d} "
          f"{np.median(chars):12.0f} / {np.percentile(chars, 90):8.0f} / {chars.max():8d}")

empty = sum(not c["text"].strip() for c in train_cases + dev_cases + test_cases)
print("\nEmpty texts:", empty)

print("\nSample:", train_cases[0]["id"], "| label", train_cases[0]["label"])
print("  start:", repr(train_cases[0]["text"][:200]))
print("  end  :", repr(train_cases[0]["text"][-200:]))


# ------------------------------------------------------------
# Class weights for the loss (inverse frequency, mean 1)
# ------------------------------------------------------------

train_label_counts = Counter(c["label"] for c in train_cases)

if CLASS_BALANCING:
    CLASS_WEIGHTS = torch.tensor([len(train_cases) / (2 * max(train_label_counts[k], 1)) for k in (0, 1)])
else:
    CLASS_WEIGHTS = torch.ones(2)

print("\nClass weights:", [round(w, 3) for w in CLASS_WEIGHTS.tolist()])

split    cases  label 0  label 1     chars (median / 90% / max)
train    37387    22067    15320        13023 /    38996 /  2492701
dev       1988      994      994        16700 /    40495 /   260198
test      1517        0        0        17320 /    42859 /   233000

Empty texts: 0

Sample: 2020_1 | label 0

Class weights: [0.847, 1.22]


### CELL 6 · Tokenize (only the part the model reads) + chunks

**In simple words:** Judgments are long and the decision is written at the end, so only the **last 1,536 tokens** are kept and cut into 3 chunks of 512.

**How it works:** The chunks are cut from the end, so the last chunk always ends exactly where the judgment ends.

In [6]:
# ============================================================
# CELL 6 — TOKENIZE (only the part the model reads) + CHUNKS
#
# DOC_PART = "end": the last DOC_TOKENS tokens, cut into chunks of
# CHUNK_LEN FROM THE END, so the last chunk always ends where the
# document ends.
# ============================================================

def split_chunks(ids, from_end):

    if from_end:
        return [ids[max(0, i - CHUNK_LEN):i] for i in range(len(ids), 0, -CHUNK_LEN)][::-1]

    return [ids[i:i + CHUNK_LEN] for i in range(0, len(ids), CHUNK_LEN)]


def make_chunks(text):
    """List of token-id arrays (at most MAX_CHUNKS, each at most CHUNK_LEN)."""

    if DOC_PART == "end":
        chunks = split_chunks(tail_tokens(text, DOC_TOKENS), from_end=True)

    elif DOC_PART == "start":
        chunks = split_chunks(head_tokens(text, DOC_TOKENS), from_end=False)

    else:  # start_end
        half = DOC_TOKENS // 2
        start = head_tokens(text, half)
        end = tail_tokens(text, half)

        if len(start) < half or len(end) < half:          # short document: read all of it once
            chunks = split_chunks(head_tokens(text, DOC_TOKENS), from_end=False)
        else:
            chunks = split_chunks(start, from_end=False) + split_chunks(end, from_end=True)

    chunks = [c for c in chunks if len(c) > 0] or [[UNK_ID]]      # empty text -> 1 token

    return [np.array(c, dtype=np.int32) for c in chunks]


t0 = time.time()

for split, cases in [("train", train_cases), ("dev", dev_cases), ("test", test_cases)]:

    for k, case in enumerate(cases):

        case["chunks"] = make_chunks(case["text"])
        case["n_tokens"] = sum(len(c) for c in case["chunks"])

        if (k + 1) % 5000 == 0:
            print(f"  {split}: {k + 1}/{len(cases)} cases | {time.time() - t0:.0f}s", flush=True)

print(f"Tokenized in {time.time() - t0:.0f}s")


# check: fast encoding == tokenizer.encode on a few real cases
for case in train_cases[:5]:
    piece = case["text"][-3000:]
    assert encode_fast(piece) == tokenizer.encode(piece)


n_tokens = np.array([c["n_tokens"] for c in train_cases])
print(f"\nTrain cases shorter than {DOC_TOKENS} tokens: {(n_tokens < DOC_TOKENS).mean():.1%}")
print(f"Chunks per case: {Counter(len(c['chunks']) for c in train_cases)}")
print(f"Tokens per student epoch: {STUDENT_EPOCH_DOCS * DOC_TOKENS / 1e6:.1f}M "
      f"({min(STUDENT_EPOCH_DOCS, len(train_cases))} cases)")

  train: 5000/37387 cases | 9s
  train: 10000/37387 cases | 16s
  train: 15000/37387 cases | 24s
  train: 20000/37387 cases | 33s
  train: 25000/37387 cases | 41s
  train: 30000/37387 cases | 47s
  train: 35000/37387 cases | 55s
Tokenized in 66s

Train cases shorter than 1536 tokens: 15.5%
Chunks per case: Counter({3: 33517, 2: 2117, 1: 1753})
Tokens per student epoch: 18.4M (12000 cases)


### CELL 7 · Metrics + decision threshold

**In simple words:** The scores (macro-F1, AUC) and the decision threshold: the probability above which we predict "accepted", chosen on dev.

In [7]:
# ============================================================
# CELL 7 — METRICS + DECISION THRESHOLD
# ============================================================

def binary_scores(labels, preds):
    """Macro P / R / F1 over the 2 classes (the competition metric is macro-F1)."""

    p, r, f, _ = precision_recall_fscore_support(labels, preds, labels=[0, 1], average="macro", zero_division=0)

    return {"precision": float(p), "recall": float(r), "f1": float(f)}


def best_threshold(labels, probs):
    """Threshold on P(label = 1) that gives the best macro-F1."""

    best_t, best_f1 = 0.5, -1.0

    # fixed grid + quantiles of the probabilities (works even when they are all close together)
    candidates = np.unique(np.concatenate([np.arange(0.05, 0.951, 0.01), np.quantile(probs, np.linspace(0.02, 0.98, 97))]))

    for t in candidates:

        f1 = f1_score(labels, (probs >= t).astype(int), labels=[0, 1], average="macro", zero_division=0)

        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)

    return best_t, best_f1


_labels = np.array([0, 0, 1, 1])
_probs = np.array([0.2, 0.6, 0.7, 0.9])
print("check:", binary_scores(_labels, (_probs >= 0.5).astype(int)), "| best threshold:", best_threshold(_labels, _probs))

### CELL 8 · Models

**In simple words:** The two networks of the distillation run: the **teacher** (Architecture-4) and the **student** (Architecture-2 encoder), with the same case head.

**How it works:** chunk → backbone → token vectors (mix of all layers, standardized) → [attention-pooled ; mean] = chunk vector → LayerNorm → BiLSTM over the chunks → attention pooling → 2 classes.

In [8]:
# ============================================================
# CELL 8 — MODELS
#
#   TEACHER backbone: Architecture-4 hybrid prefix-LM
#   STUDENT backbone: Architecture-2 encoder (the decoder is not needed)
#   same head for both:
#     every chunk -> backbone -> token vector = learned mix of ALL layers
#                 -> standardized -> [attention-pooled ; mean] = chunk vector
#     chunks of a case -> LayerNorm -> BiLSTM over the chunks -> attention pooling -> 2 classes
#
# Layer names match the Phase-1 notebooks, so the weights load directly.
# ============================================================

# ============================================================
# Architecture-4 HYBRID PREFIX-LM (TEACHER backbone)
# ============================================================

class HybridRMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (self.weight * (xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps))).to(x.dtype)


class HybridAttention(nn.Module):
    """Used twice per block: bidirectional (prefix / encoder branch) and causal (causal branch)."""

    def __init__(self, d_model, n_heads, dropout, causal):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.causal = causal

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask):

        B, T, D = x.shape
        q, k, v = (p(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
                   for p in (self.q_proj, self.k_proj, self.v_proj))

        p = self.dropout.p if self.training else 0.0

        if self.causal:      # padding is at the end, so real tokens never see it
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=p)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=p)

        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, D))


class GatedHybridFusion(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        self.gate_proj = nn.Linear(2 * d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, encoder_output, causal_output):
        gate = torch.sigmoid(self.gate_proj(torch.cat([encoder_output, causal_output], dim=-1)))
        return self.out_proj(gate * encoder_output + (1.0 - gate) * causal_output)


class HybridSwiGLU(nn.Module):

    def __init__(self, d_model, ffn_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class HybridTransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        d, h, p, eps = cfg["d_model"], cfg["n_heads"], cfg["dropout"], cfg["norm_eps"]

        self.encoder_norm = HybridRMSNorm(d, eps)
        self.causal_norm = HybridRMSNorm(d, eps)
        self.encoder_attention = HybridAttention(d, h, p, causal=False)
        self.causal_attention = HybridAttention(d, h, p, causal=True)
        self.hybrid_fusion = GatedHybridFusion(d)
        self.ffn_norm = HybridRMSNorm(d, eps)
        self.ffn = HybridSwiGLU(d, cfg["ffn_dim"], p)

    def forward(self, x, mask, prefix):

        T = x.size(1)

        # encoder branch: the first `prefix` tokens attend to each other (both directions);
        # tokens after the prefix get 0 from this branch (as in pretraining)
        encoder_output = self.encoder_attention(self.encoder_norm(x[:, :prefix]), mask[:, None, None, :prefix])
        encoder_output = F.pad(encoder_output, (0, 0, 0, T - prefix))

        causal_output = self.causal_attention(self.causal_norm(x), None)

        x = x + self.hybrid_fusion(encoder_output, causal_output)

        return x + self.ffn(self.ffn_norm(x))


class Arch4Hybrid(nn.Module):
    """Architecture-4 HybridLM without the LM head: tokens -> hidden vectors."""

    def __init__(self, cfg):
        super().__init__()

        self.prefix_length = cfg["prefix_length"]

        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = nn.Embedding(cfg["block_size"], cfg["d_model"])
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([HybridTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = HybridRMSNorm(cfg["d_model"], cfg["norm_eps"])

    def embed(self, input_ids):
        """Token + position embeddings (the first hidden state)."""
        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        return self.embedding_dropout(self.token_embedding(input_ids) + self.position_embedding(positions))

    def forward(self, input_ids, mask, all_layers=False):
        """all_layers=False: final hidden vectors (as in Phase 1).
        all_layers=True : list of the embeddings + the output of every block (before any norm)."""

        T = input_ids.size(1)
        prefix = min(self.prefix_length, T)

        x = self.embed(input_ids)
        states = [x]

        for block in self.blocks:
            x = block(x, mask, prefix)
            states.append(x)

        if all_layers:
            return states

        return self.final_norm(x)


# ============================================================
# Architecture-2 ENCODER (STUDENT backbone)
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_heads, dropout):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, attn_mask):

        q, k, v = self.split_heads(self.q_proj(x)), self.split_heads(self.k_proj(x)), self.split_heads(self.v_proj(x))

        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                             dropout_p=self.dropout if self.training else 0.0)
        B, H, T, d = out.shape

        return self.out_proj(out.transpose(1, 2).reshape(B, T, H * d))


class PositionalEmbedding(nn.Module):

    def __init__(self, max_len, d_model, dropout):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return self.dropout(x + self.embedding(positions).unsqueeze(0))


class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))


class EncoderBlock(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, attn_mask):
        x = x + self.self_attention(self.norm1(x), attn_mask)
        return x + self.feed_forward(self.norm2(x))


class Arch2Encoder(nn.Module):
    """Architecture-2 encoder (the decoder is not needed): bidirectional, learned positions."""

    def __init__(self, cfg):
        super().__init__()

        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = PositionalEmbedding(cfg["src_len"], cfg["d_model"], cfg["dropout"])
        self.layers = nn.ModuleList([
            EncoderBlock(cfg["d_model"], cfg["n_heads"], cfg["ffn_dim"], cfg["dropout"])
            for _ in range(cfg["encoder_layers"])
        ])
        self.final_norm = nn.LayerNorm(cfg["d_model"])

    def embed(self, input_ids):
        """Token + position embeddings (the first hidden state)."""
        return self.position_embedding(self.token_embedding(input_ids))

    def forward(self, input_ids, mask, all_layers=False):
        """all_layers=False: final hidden vectors (as in Phase 1).
        all_layers=True : list of the embeddings + the output of every layer (before any norm)."""

        attn_mask = mask[:, None, None, :]                    # padding is never attended to
        x = self.embed(input_ids)
        states = [x]

        for layer in self.layers:
            x = layer(x, attn_mask)
            states.append(x)

        if all_layers:
            return states

        return self.final_norm(x)


# ============================================================
# What trains in the Phase-1 model: only the TOKEN EMBEDDINGS (or nothing)
# ============================================================

def backbone_blocks(backbone):
    """The transformer blocks of a Phase-1 model, bottom -> top
    (Architecture-2 calls them layers, Architecture-4 blocks)."""

    return list(backbone.layers if hasattr(backbone, "layers") else backbone.blocks)


def set_trainable(model, embeddings):
    """Freeze every Phase-1 weight; then unfreeze the token embeddings if `embeddings`.
    The blocks always run without gradients; the embeddings learn through the first hidden state."""

    for p in model.backbone.parameters():
        p.requires_grad = False

    model.backbone.token_embedding.weight.requires_grad = embeddings
    model.train_backbone = False
    model.train_embeddings = embeddings


# ============================================================
# Standardization with RUNNING statistics (no learned weights)
#   x -> (x - mean) / std
# The statistics start from the frozen Phase-1 model (CELL 12) and then
# slowly follow the features while the unfrozen blocks change them (fixed statistics
# would make the features drift away from what the head has learned).
# ============================================================

class RunningStandardize(nn.Module):

    def __init__(self, dim, momentum=0.01):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("mean", torch.zeros(dim))
        self.register_buffer("std", torch.ones(dim))

    def forward(self, x, mask):
        """x [..., dim], mask [...] (True = real value)."""

        if self.training:
            with torch.no_grad():
                values = x[mask]
                if len(values) > 1:
                    self.mean.mul_(1 - self.momentum).add_(self.momentum * values.mean(dim=0))
                    self.std.mul_(1 - self.momentum).add_(self.momentum * values.std(dim=0).clamp(min=1e-4))

        return (x - self.mean) / self.std


# ============================================================
# Case classifier (hierarchical: tokens -> chunks -> case)
# ============================================================

class CaseClassifier(nn.Module):

    def __init__(self, backbone, d_model, n_states):
        super().__init__()

        self.backbone = backbone
        self.d_model = d_model
        self.kd_alpha = 0.0                 # > 0: distillation loss (the student, STEP 3)
        self.train_backbone = True          # False = the Phase-1 model runs without gradients
        self.train_embeddings = False       # True = the token embeddings train (through the first hidden state)

        # Layer mix: token vector = sum_l w_l * LayerNorm(h_l) over the embeddings and every
        # block, w = softmax(layer_logits). The last layer alone hides the words that decide a
        # case (Architecture-2: frozen probe AUC 0.56 vs 0.71 for word counts); the lower layers
        # keep them. Start values: CELL 12 (per-layer probe); then learned with the head.
        self.layer_logits = nn.Parameter(torch.zeros(n_states))

        # Token vectors of a language model share one big direction (chunks of
        # different cases have cosine ~0.97); what tells cases apart is small.
        # Standardizing every feature (minus its mean over train tokens, divided
        # by its std) keeps only that part. Statistics: CELL 12.
        self.token_standardize = RunningStandardize(d_model)

        # token attention: LEARNS which tokens of a chunk matter (an average over
        # 512 tokens drowns the few words that decide a case)
        self.token_attention = nn.Sequential(nn.Linear(d_model, 128), nn.Tanh(), nn.Linear(128, 1))

        # chunk vectors are standardized again (statistics over train chunks, CELL 12):
        # this is what lets the head see the small differences between cases
        self.chunk_standardize = RunningStandardize(2 * d_model)

        self.vector_norm = nn.LayerNorm(2 * d_model)                  # [attention-pooled ; mean] -> 2 * d_model
        self.dropout = nn.Dropout(HEAD_DROPOUT)

        self.lstm = nn.LSTM(2 * d_model, LSTM_HIDDEN, batch_first=True, bidirectional=True)

        self.attention = nn.Sequential(nn.Linear(2 * LSTM_HIDDEN, 128), nn.Tanh(), nn.Linear(128, 1))
        self.classifier = nn.Linear(2 * LSTM_HIDDEN, 2)
        nn.init.zeros_(self.classifier.weight)          # the DEEP path starts at 0 ...
        nn.init.zeros_(self.classifier.bias)

        # ... and the LINEAR path (per case: mean over its chunks of the mean-pooled, standardized
        # token vectors -> 2 classes) starts as the logistic regression of CELL 11: training
        # starts from what the frozen features already give and only has to improve on it
        self.probe_classifier = nn.Linear(d_model, 2)
        nn.init.zeros_(self.probe_classifier.weight)
        nn.init.zeros_(self.probe_classifier.bias)

        self.register_buffer("class_weights", CLASS_WEIGHTS.clone().float())

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("backbone.")]

    def layer_weights(self):
        return torch.softmax(self.layer_logits.float(), dim=0)

    def token_states(self, ids, mask):
        """[n_chunks, T] ids -> [n_chunks, T, D] float32: mix of the LayerNorm-ed hidden states of all layers."""

        with torch.set_grad_enabled(torch.is_grad_enabled() and self.train_backbone):
            with autocast():
                states = self.backbone(ids, mask, all_layers=True)

        if self.train_embeddings and torch.is_grad_enabled():
            # the token embeddings learn through the first hidden state directly
            # (the blocks stay frozen and ran without gradients: fast)
            with autocast():
                states[0] = self.backbone.embed(ids)

        mixed = 0.0
        for w, h in zip(self.layer_weights(), states):
            mixed = mixed + w * F.layer_norm(h.float(), h.shape[-1:])

        return mixed

    def chunk_vectors(self, chunks):
        """List of token-id arrays -> [n_chunks, 2D] float32.
        Chunk vector = [attention-pooled ; mean] of its standardized token vectors."""

        max_len = max(len(c) for c in chunks)
        ids = torch.full((len(chunks), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(chunks), max_len), dtype=torch.bool)

        for row, c in enumerate(chunks):
            ids[row, :len(c)] = torch.from_numpy(c.astype(np.int64))
            mask[row, :len(c)] = True

        ids, mask = ids.to(DEVICE), mask.to(DEVICE)

        hidden = self.token_states(ids, mask)                         # [n_chunks, T, D], mix of all layers

        # do not pool the first POOL_SKIP_TOKENS tokens (unless the chunk is very short)
        pool_mask = mask.clone()
        long_chunk = mask.sum(dim=1) > 2 * POOL_SKIP_TOKENS
        pool_mask[long_chunk, :POOL_SKIP_TOKENS] = False

        z = self.token_standardize(hidden, pool_mask)                  # standardized token vectors

        scores = self.token_attention(z).squeeze(-1).masked_fill(~pool_mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)                         # [n_chunks, T]
        attended = (weights.unsqueeze(-1) * z).sum(dim=1)

        m = pool_mask.unsqueeze(-1)
        mean = (z * m).sum(dim=1) / m.sum(dim=1)

        return torch.cat([attended, mean], dim=-1)

    def classify(self, case_vectors):
        """
        case_vectors: list of [n_chunks_i, 2D] tensors (one per case, chunks in document order)
        returns logits [n_cases, 2] and attention weights [n_cases, max_chunks]
        """

        n_chunks = [len(v) for v in case_vectors]
        K = max(n_chunks)

        x = torch.zeros(len(case_vectors), K, case_vectors[0].size(-1), device=DEVICE)
        mask = torch.zeros(len(case_vectors), K, dtype=torch.bool, device=DEVICE)

        for i, v in enumerate(case_vectors):
            x[i, :len(v)] = v.to(DEVICE)
            mask[i, :len(v)] = True

        x = self.chunk_standardize(x, mask)

        # linear path: per case, the mean over its chunks of the (standardized) mean-pooled token vectors
        d = x.size(-1) // 2
        m = mask.unsqueeze(-1).float()
        case_mean = (x[..., d:] * m).sum(dim=1) / m.sum(dim=1)

        x = self.dropout(self.vector_norm(x))

        lengths = torch.tensor(n_chunks)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=K)    # [n, K, 2H]

        scores = self.attention(out).squeeze(-1).masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)                                            # [n, K]

        case_vector = (weights.unsqueeze(-1) * out).sum(dim=1)

        return self.probe_classifier(case_mean) + self.classifier(self.dropout(case_vector)), weights

    def forward(self, cases):

        all_chunks = [c for case in cases for c in case["chunks"]]
        vectors = self.chunk_vectors(all_chunks)

        case_vectors, k = [], 0
        for case in cases:
            n = len(case["chunks"])
            case_vectors.append(vectors[k:k + n])
            k += n

        return self.classify(case_vectors)

    def loss(self, cases):
        """Teacher: cross-entropy with the gold labels.
        Student: (1 - KD_ALPHA) * cross-entropy + KD_ALPHA * T^2 * KL(teacher || student)."""

        logits, _ = self(cases)
        logits = logits.float()
        labels = torch.tensor([case["label"] for case in cases], device=DEVICE)

        hard = F.cross_entropy(logits, labels, weight=self.class_weights)

        if self.kd_alpha == 0:
            return hard

        p1 = torch.tensor([case["teacher_prob"] for case in cases], device=DEVICE).clamp(1e-4, 1 - 1e-4)
        teacher_log_probs = torch.log(torch.stack([1 - p1, p1], dim=1))

        T = KD_TEMPERATURE
        teacher_soft = torch.softmax(teacher_log_probs / T, dim=1)
        soft = F.kl_div(F.log_softmax(logits / T, dim=1), teacher_soft, reduction="batchmean") * T * T

        return (1 - self.kd_alpha) * hard + self.kd_alpha * soft


### CELL 9 · Load both Phase-1 checkpoints + model builders

**In simple words:** Load the Phase-1 weights of the teacher and of the student, and define how each network is built.

**How it works:** Each builder loads the weights, adds the technique and checks which weights train.

In [9]:
# ============================================================
# CELL 9 — LOAD BOTH PHASE-1 CHECKPOINTS + MODEL BUILDERS
# ============================================================

def find_checkpoint(path, file_name, required_key):
    """Use `path` if it exists, else search /kaggle/input for `file_name` whose config has `required_key`."""

    if Path(path).exists():
        return Path(path)

    for candidate in sorted(Path("/kaggle/input").rglob(file_name)):
        try:
            ckpt = torch.load(candidate, map_location="cpu", weights_only=False)
            if required_key in ckpt.get("model_config", ckpt.get("config", {})):
                print("Found:", candidate)
                return candidate
        except Exception:
            pass

    raise FileNotFoundError(f"{path} not found and no matching {file_name} under /kaggle/input")


# ---------------- teacher: Architecture-4 ----------------
TEACHER_CHECKPOINT = find_checkpoint(TEACHER_CHECKPOINT, "best.pt", "prefix_length")
phase1 = torch.load(TEACHER_CHECKPOINT, map_location="cpu", weights_only=False)

TEACHER_CONFIG = dict(phase1["model_config"])
TEACHER_CONFIG.setdefault("norm_eps", 1e-5)
TEACHER_CONFIG["dropout"] = BACKBONE_DROPOUT
if PREFIX_LENGTH is not None:
    TEACHER_CONFIG["prefix_length"] = PREFIX_LENGTH

# weights without the LM head (tied to the embedding, not needed here)
TEACHER_WEIGHTS = {
    key.replace("_orig_mod.", ""): value
    for key, value in phase1["model_state_dict"].items()
    if not key.replace("_orig_mod.", "").startswith("lm_head.")
}

# ---------------- student: Architecture-2 encoder ----------------
STUDENT_CHECKPOINT = find_checkpoint(STUDENT_CHECKPOINT, "best_encoder_decoder.pt", "encoder_layers")
phase1 = torch.load(STUDENT_CHECKPOINT, map_location="cpu", weights_only=False)

STUDENT_CONFIG = dict(phase1["config"])
STUDENT_CONFIG["dropout"] = BACKBONE_DROPOUT

# encoder weights only (the decoder of Architecture-2 is not needed)
STUDENT_WEIGHTS = {}
for key, value in phase1["model_state_dict"].items():
    key = key.replace("_orig_mod.", "")
    if key.startswith("encoder."):
        STUDENT_WEIGHTS[key[len("encoder."):]] = value

del phase1

print("Teacher config:", {k: TEACHER_CONFIG[k] for k in ("d_model", "n_layers", "n_heads", "ffn_dim", "block_size", "prefix_length")})
print("Student config:", {k: STUDENT_CONFIG[k] for k in ("d_model", "encoder_layers", "n_heads", "ffn_dim", "src_len")})

for config in (TEACHER_CONFIG, STUDENT_CONFIG):
    assert config["vocab_size"] == VOCAB_SIZE, "checkpoint was trained with a different tokenizer"
assert CHUNK_LEN <= TEACHER_CONFIG["block_size"] and CHUNK_LEN <= STUDENT_CONFIG["src_len"]


def build(backbone, weights, d_model):
    """Phase-1 backbone (every weight frozen at the start) + new head."""

    backbone.load_state_dict(weights, strict=True)          # every name and shape must match

    for p in backbone.parameters():
        p.requires_grad = False

    # the hidden states of all layers; the last one (after the final norm) = the Phase-1 output
    test_ids = torch.randint(4, VOCAB_SIZE, (2, 24))
    test_mask = torch.ones(2, 24, dtype=torch.bool)
    test_mask[1, 16:] = False

    backbone.eval()
    with torch.no_grad():
        states = backbone(test_ids, test_mask, all_layers=True)
        assert len(states) == len(backbone_blocks(backbone)) + 1
        assert torch.allclose(backbone.final_norm(states[-1]), backbone(test_ids, test_mask), atol=1e-5)
    backbone.train()

    return CaseClassifier(backbone, d_model, len(backbone_blocks(backbone)) + 1).to(DEVICE)


def build_teacher():
    return build(Arch4Hybrid(TEACHER_CONFIG), TEACHER_WEIGHTS, TEACHER_CONFIG["d_model"])


def build_student():
    return build(Arch2Encoder(STUDENT_CONFIG), STUDENT_WEIGHTS, STUDENT_CONFIG["d_model"])


def backbone_size(model):
    """Weights of the Phase-1 model that are used (the Architecture-2 decoder is not)."""
    return sum(p.numel() for p in model.backbone.parameters())


# sizes (built on the CPU once, then freed)
_t, _s = Arch4Hybrid(TEACHER_CONFIG), Arch2Encoder(STUDENT_CONFIG)
TEACHER_BACKBONE_SIZE = sum(p.numel() for p in _t.parameters())
STUDENT_BACKBONE_SIZE = sum(p.numel() for p in _s.parameters())
del _t, _s

print(f"\nTeacher backbone: {TEACHER_BACKBONE_SIZE:,} weights | student backbone: {STUDENT_BACKBONE_SIZE:,} "
      f"({TEACHER_BACKBONE_SIZE / STUDENT_BACKBONE_SIZE:.1f}x smaller)")


Teacher config: {'d_model': 768, 'n_layers': 16, 'n_heads': 12, 'ffn_dim': 2048, 'block_size': 2048, 'prefix_length': 512}
Student config: {'d_model': 768, 'encoder_layers': 6, 'n_heads': 12, 'ffn_dim': 3072, 'src_len': 1024}

Teacher backbone: 205,496,064 weights | student backbone: 67,894,272 (3.0x smaller)


### CELL 10 · Prediction + evaluation (any model: teacher or student)

**In simple words:** Turn every case into chunk vectors once (in big batches), then the head gives P(accepted).

In [10]:
# ============================================================
# CELL 10 — PREDICTION + EVALUATION (any model: teacher or student)
#
# Chunk vectors are computed once per case (in big batches), then
# the classifier turns them into P(label = 1).
# ============================================================

@torch.no_grad()
def case_vectors(model, cases):
    """List of [n_chunks, 2D] CPU tensors, one per case."""

    model.eval()

    all_chunks = [c for case in cases for c in case["chunks"]]
    vectors = []

    for k in range(0, len(all_chunks), EVAL_CHUNKS):
        vectors.append(model.chunk_vectors(all_chunks[k:k + EVAL_CHUNKS]).cpu())

    vectors = torch.cat(vectors)

    out, k = [], 0
    for case in cases:
        n = len(case["chunks"])
        out.append(vectors[k:k + n])
        k += n

    model.train()

    return out


@torch.no_grad()
def classify_vectors(model, vectors, batch=256):
    """Chunk vectors -> (P(label = 1) [n_cases], attention weights list)."""

    model.eval()

    probs, weights = [], []

    for k in range(0, len(vectors), batch):

        logits, w = model.classify(vectors[k:k + batch])

        probs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu())
        weights.extend([w[i, :len(v)].cpu().numpy() for i, v in enumerate(vectors[k:k + batch])])

    model.train()

    return torch.cat(probs).numpy(), weights


def evaluate(model, cases):
    """Dev scores at threshold 0.5 and at the best threshold."""

    labels = np.array([c["label"] for c in cases])
    probs, _ = classify_vectors(model, case_vectors(model, cases))

    threshold, _ = best_threshold(labels, probs)

    return {
        "at_0.5": binary_scores(labels, (probs >= 0.5).astype(int)),
        "threshold": threshold,
        "at_threshold": binary_scores(labels, (probs >= threshold).astype(int)),
        "auc": float(roc_auc_score(labels, probs)) if len(set(labels)) == 2 else 0.0,
    }, probs

### CELL 11 · Prepare a model before training (teacher and student)

**In simple words:** Before training (teacher and student): probe every layer, store feature statistics, and start the classifier from a logistic regression, so training begins from a sensible model rather than a random one.

In [11]:
# ============================================================
# CELL 11 — PREPARE A MODEL BEFORE TRAINING (teacher and student)
#
#   1. linear probe on EVERY layer of the FROZEN Phase-1 model
#      (case vector = mean of the LayerNorm-ed token vectors of that layer
#      -> logistic regression on PROBE_DOCS train cases -> dev AUC / macro-F1).
#      The layer mix of the head STARTS from these scores.
#   2. token statistics of the mixed token vectors (STATS_DOCS train cases)
#      -> stored in the model; then the same for the chunk vectors
#   3. the LINEAR path of the classifier starts as a logistic regression on the
#      mixed features (from the per-layer means of step 1, no new pass)
# ============================================================

from sklearn.linear_model import LogisticRegression


def chunk_batches(chunks):
    """Padded batches of EVAL_CHUNKS chunks: (ids, attention mask, pooling mask without the first tokens)."""

    for k in range(0, len(chunks), EVAL_CHUNKS):

        batch = chunks[k:k + EVAL_CHUNKS]
        max_len = max(len(c) for c in batch)
        ids = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(batch), max_len), dtype=torch.bool)

        for row, c in enumerate(batch):
            ids[row, :len(c)] = torch.from_numpy(c.astype(np.int64))
            mask[row, :len(c)] = True

        pool_mask = mask.clone()
        long_chunk = mask.sum(dim=1) > 2 * POOL_SKIP_TOKENS
        pool_mask[long_chunk, :POOL_SKIP_TOKENS] = False

        yield ids.to(DEVICE), mask.to(DEVICE), pool_mask.to(DEVICE)


@torch.no_grad()
def layer_means(model, cases):
    """[n_cases, N_STATES, D]: per case and layer, the mean LayerNorm-ed token vector (averaged over its chunks)."""

    model.eval()

    chunks = [c for case in cases for c in case["chunks"]]
    per_chunk = []

    for ids, mask, pool_mask in chunk_batches(chunks):

        with autocast():
            states = model.backbone(ids, mask, all_layers=True)

        m = pool_mask.unsqueeze(-1)
        per_chunk.append(torch.stack(
            [(F.layer_norm(h.float(), h.shape[-1:]) * m).sum(dim=1) / m.sum(dim=1) for h in states], dim=1).cpu())

    model.train()

    per_chunk = torch.cat(per_chunk)

    out, k = [], 0
    for case in cases:
        n = len(case["chunks"])
        out.append(per_chunk[k:k + n].mean(dim=0))
        k += n

    return torch.stack(out)


def prepare(model, name):
    """Per-layer probe -> start of the layer mix; feature statistics -> stored in the model."""

    t0 = time.time()

    n_states = len(model.layer_logits)
    layer_names = ["embeddings"] + [f"layer {i}" for i in range(1, n_states)]

    # ---------------- 1. probe every layer ----------------
    probe_cases = random.Random(SEED + 1).sample(train_cases, min(PROBE_DOCS, len(train_cases)))

    L_probe = layer_means(model, probe_cases)
    L_dev = layer_means(model, dev_cases)
    y_probe = np.array([c["label"] for c in probe_cases])
    y_dev = np.array([c["label"] for c in dev_cases])

    print(f"{name}: linear probe on the frozen Phase-1 model ({len(probe_cases)} train cases -> dev)")
    print(f"  {'layer':12s} {'dev AUC':>8s} {'macro-F1':>9s}")

    probe_results = {}

    for l, layer in enumerate(layer_names):

        mu = L_probe[:, l].mean(dim=0)
        sd = L_probe[:, l].std(dim=0).clamp(min=1e-6)
        X_train = ((L_probe[:, l] - mu) / sd).numpy()
        X_dev = ((L_dev[:, l] - mu) / sd).numpy()

        best = None
        for C in (0.01, 0.1):
            clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
            clf.fit(X_train, y_probe)
            p_dev = clf.predict_proba(X_dev)[:, 1]
            auc_l = float(roc_auc_score(y_dev, p_dev))
            if best is None or auc_l > best["auc"]:
                t, f1 = best_threshold(y_dev, p_dev)
                best = {"auc": auc_l, "f1": f1, "threshold": t, "C": C}

        probe_results[layer] = best
        print(f"  {layer:12s} {best['auc']:8.4f} {best['f1']:9.4f}")

    best_layer = max(probe_results, key=lambda n: probe_results[n]["auc"])

    # the layer mix starts from the probe, concentrated on the best layers:
    # weight ~ exp((AUC - best AUC) / 0.02)
    best_auc = probe_results[best_layer]["auc"]
    with torch.no_grad():
        model.layer_logits.copy_(torch.tensor(
            [(probe_results[n]["auc"] - best_auc) / 0.02 for n in layer_names]).to(DEVICE))

    print(f"  best frozen layer: {best_layer} (dev AUC {probe_results[best_layer]['auc']:.4f})")

    # ---------------- 2. token statistics of the mixed token vectors ----------------
    stat_cases = random.Random(SEED).sample(train_cases, min(STATS_DOCS, len(train_cases)))
    stat_chunks = [c for case in stat_cases for c in case["chunks"]]

    total = torch.zeros(model.d_model, dtype=torch.float64, device=DEVICE)
    total_sq = torch.zeros_like(total)
    count = 0

    model.eval()

    with torch.no_grad():
        for ids, mask, pool_mask in chunk_batches(stat_chunks):
            hidden = model.token_states(ids, mask)
            m = pool_mask.unsqueeze(-1)
            total += (hidden * m).sum(dim=(0, 1)).double()
            total_sq += (hidden.pow(2) * m).sum(dim=(0, 1)).double()
            count += int(pool_mask.sum())

    model.train()

    mean = (total / count).float()
    std = (total_sq / count - (total / count).pow(2)).clamp(min=1e-8).sqrt().float().clamp(min=1e-4)

    model.token_standardize.mean.copy_(mean)
    model.token_standardize.std.copy_(std)

    stat_vectors = torch.cat(case_vectors(model, stat_cases))
    chunk_mean = stat_vectors.mean(dim=0)
    chunk_std = stat_vectors.std(dim=0).clamp(min=1e-4)
    model.chunk_standardize.mean.copy_(chunk_mean.to(DEVICE))
    model.chunk_standardize.std.copy_(chunk_std.to(DEVICE))

    # ---------------- 3. the linear path starts as a logistic regression ----------------
    # case feature = mean over its chunks of the mean-pooled, MIXED token vectors, standardized
    # like the chunk vectors (exactly what the linear path sees; the mix and the means are
    # linear -> computed from the per-layer means of step 1, no new pass)
    w = model.layer_weights().detach().cpu()
    d = model.d_model

    def mixed_case_features(L):
        mixed = (L * w[None, :, None]).sum(dim=1)                       # mean of the mixed token vectors
        z = (mixed - mean.cpu()) / std.cpu()                             # token standardization
        return ((z - chunk_mean[d:]) / chunk_std[d:]).numpy()            # chunk standardization

    F_probe, F_dev = mixed_case_features(L_probe), mixed_case_features(L_dev)

    best = None
    for C in (0.01, 0.1):
        clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
        clf.fit(F_probe, y_probe)
        p_dev = clf.predict_proba(F_dev)[:, 1]
        if best is None or roc_auc_score(y_dev, p_dev) > best[0]:
            best = (float(roc_auc_score(y_dev, p_dev)), clf, p_dev)

    start_auc, clf, p_dev = best
    start_f1 = best_threshold(y_dev, p_dev)[1]

    # logits = [0, w x + b]
    with torch.no_grad():
        model.probe_classifier.weight.zero_()
        model.probe_classifier.bias.zero_()
        model.probe_classifier.weight[1] = torch.tensor(clf.coef_[0], dtype=torch.float32)
        model.probe_classifier.bias[1] = float(clf.intercept_[0])

    print(f"  feature statistics from {count:,} tokens")
    print(f"  {name} STARTS as a logistic regression on the mixed features: dev AUC {start_auc:.4f}, "
          f"macro-F1 {start_f1:.4f} | {time.time() - t0:.0f}s")

    return {"by_layer": probe_results, "best_layer": best_layer, "best": probe_results[best_layer],
            "layer_names": layer_names, "start": {"auc": start_auc, "f1": start_f1}}


### CELL 12 · Optimizer + training function (teacher and student)

**In simple words:** Optimizer and training loop for teacher and student.

**How it works:** Epoch 0 = the starting logistic regression; every later epoch trains the head + the token embeddings; the best dev epoch is kept.

In [12]:
# ============================================================
# CELL 12 — OPTIMIZER + TRAINING FUNCTION (teacher and student)
#   group 0 = token embeddings of the Phase-1 model, group 1 = head
#   epoch 0 = the start (logistic regression); every epoch after it trains
#   the head + the token embeddings; the best epoch on dev is kept
# ============================================================

def make_optimizer(model):

    groups = [{"params": model.head_parameters(), "lr": LR_HEAD, "weight_decay": 0.01}]

    if model.train_embeddings:
        groups.insert(0, {"params": [model.backbone.token_embedding.weight], "lr": LR_EMBEDDINGS, "weight_decay": 0.0})

    return torch.optim.AdamW(groups, betas=(0.9, 0.999))


def set_lr(optimizer, base_lrs, step, total_steps):
    """Linear warmup, then linear decay to 0."""

    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

    if step < warmup_steps:
        factor = (step + 1) / warmup_steps
    else:
        factor = max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

    for group, base_lr in zip(optimizer.param_groups, base_lrs):
        group["lr"] = base_lr * factor


def trainable_state(model):
    """The weights that train (head + token embeddings) + the running feature statistics."""

    state = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
    state.update({name: b.detach().cpu().clone() for name, b in model.named_buffers() if "standardize" in name})

    return state


def restore(model, state, phase1_weights):
    """Back to a saved state: first every Phase-1 weight as in the checkpoint,
    then the weights that had been trained up to that epoch."""

    model.backbone.load_state_dict(phase1_weights, strict=True)
    model.load_state_dict(state, strict=False)


def train_model(model, name, phase1_weights, cases, epochs, epoch_docs, batch_docs, grad_accum):
    """
    Trains on `cases` (TRAIN cases only). Every epoch uses epoch_docs cases (shuffled once and
    walked through, so every case is used before any repeats). After every epoch: dev macro-F1
    (at the best threshold); the best epoch (epoch 0 = the start) is kept.
    """

    set_trainable(model, TRAIN_EMBEDDINGS)

    docs_per_epoch = min(epoch_docs, len(cases))
    updates_per_epoch = math.ceil(docs_per_epoch / (batch_docs * grad_accum))
    total_steps = epochs * updates_per_epoch

    optimizer = make_optimizer(model)
    base_lrs = [group["lr"] for group in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    rng = random.Random(SEED)
    order = list(range(len(cases)))
    rng.shuffle(order)
    cursor = 0

    history = []
    step = 0
    t_start = time.time()

    # epoch 0 = the start (logistic regression on the frozen features): kept if no epoch beats it
    dev_scores, _ = evaluate(model, dev_cases)
    best_score, best_epoch, best_state = dev_scores["at_threshold"]["f1"], 0, trainable_state(model)
    history.append({"epoch": 0, "dev": dev_scores})
    print(f"{name} epoch 0 (start = logistic regression) | dev macro-F1 @{dev_scores['threshold']:.2f} "
          f"{dev_scores['at_threshold']['f1']:.4f} | AUC {dev_scores['auc']:.4f}", flush=True)

    model.train()

    for epoch in range(1, epochs + 1):

        # the next docs_per_epoch cases of the shuffled order
        if cursor + docs_per_epoch > len(order):
            rng.shuffle(order)
            cursor = 0

        epoch_cases = [cases[i] for i in order[cursor:cursor + docs_per_epoch]]

        cursor += docs_per_epoch

        batches = [epoch_cases[k:k + batch_docs] for k in range(0, len(epoch_cases), batch_docs)]

        total_loss, skipped = 0.0, 0
        optimizer.zero_grad(set_to_none=True)

        for b, batch in enumerate(batches):

            loss = model.loss(batch)
            scaler.scale(loss / grad_accum).backward()
            total_loss += loss.item()

            if (b + 1) % grad_accum == 0 or b + 1 == len(batches):

                set_lr(optimizer, base_lrs, step, total_steps)

                scaler.unscale_(optimizer)

                # clip the token embeddings and the head separately
                for group in optimizer.param_groups:
                    torch.nn.utils.clip_grad_norm_(group["params"], GRAD_CLIP)

                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped += scaler.get_scale() < scale_before

                optimizer.zero_grad(set_to_none=True)
                step += 1

            if (b + 1) % 500 == 0:
                print(f"    batch {b + 1}/{len(batches)} | loss {total_loss / (b + 1):.4f} | "
                      f"{(time.time() - t_start) / 60:.1f} min", flush=True)

        dev_scores, _ = evaluate(model, dev_cases)

        log = {"epoch": epoch, "loss": total_loss / len(batches), "dev": dev_scores, "skipped_steps": int(skipped)}

        line = (f"{name} epoch {epoch}/{epochs} | loss {log['loss']:.4f} | "
                f"dev macro-F1 @0.5 {dev_scores['at_0.5']['f1']:.4f} | "
                f"@{dev_scores['threshold']:.2f} {dev_scores['at_threshold']['f1']:.4f} | AUC {dev_scores['auc']:.4f}")

        stop = False

        if dev_scores["at_threshold"]["f1"] > best_score:
            best_score, best_epoch, best_state = dev_scores["at_threshold"]["f1"], epoch, trainable_state(model)
            line += "  <- best"
        elif epoch - best_epoch >= PATIENCE:
            stop = True

        if skipped:
            line += f" | skipped {skipped}"
        line += f" | {(time.time() - t_start) / 60:.1f} min"

        print(line, flush=True)
        history.append(log)

        mix = " ".join(f"{w:.2f}" for w in model.layer_weights().detach().tolist())
        print(f"    layer mix (embeddings, layer 1..): {mix}")


        if stop:
            print(f"Early stop: no improvement for {PATIENCE} epochs.")
            break

    restore(model, best_state, phase1_weights)    # back to the best epoch

    return history, best_epoch, best_score


### CELL 13 · Step 1: the teacher (Architecture-4), trained on train with the gold labels

**In simple words:** **Step 1:** train the teacher (Architecture-4) on the train cases with the gold labels.

In [13]:
# ============================================================
# CELL 13 — STEP 1: THE TEACHER (Architecture-4), trained on TRAIN with the gold labels
#   (or loaded from TEACHER_TRAINED_PATH, see CELL 2)
# ============================================================

for name in ["teacher", "student", "model", "optimizer", "scaler"]:
    globals().pop(name, None)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

teacher = build_teacher()
set_trainable(teacher, TRAIN_EMBEDDINGS)
TEACHER_PROBE = prepare(teacher, "Teacher")

t0 = time.time()

if TEACHER_TRAINED_PATH is not None:

    state = torch.load(TEACHER_TRAINED_PATH, map_location="cpu", weights_only=False)
    restore(teacher, state, TEACHER_WEIGHTS)

    assert "backbone.token_embedding.weight" in state or not TRAIN_EMBEDDINGS, \
        "TEACHER_TRAINED_PATH must be a teacher_trainable.pt saved by THIS notebook (v3)"
    teacher_history, teacher_best_epoch = [], 0
    teacher_scores, _ = evaluate(teacher, dev_cases)
    print(f"Teacher loaded from {TEACHER_TRAINED_PATH}")

else:

    print(f"\nTeacher | train {len(train_cases)} cases ({min(TEACHER_EPOCH_DOCS, len(train_cases))} per epoch) -> dev {len(dev_cases)}")
    teacher_history, teacher_best_epoch, _ = train_model(
        teacher, "Teacher", TEACHER_WEIGHTS, train_cases, TEACHER_EPOCHS, TEACHER_EPOCH_DOCS,
        TEACHER_BATCH_DOCS, TEACHER_GRAD_ACCUM)
    teacher_scores, _ = evaluate(teacher, dev_cases)

teacher_minutes = (time.time() - t0) / 60
teacher_peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

print(f"\n✅ Teacher dev macro-F1 {teacher_scores['at_threshold']['f1']:.4f} (threshold {teacher_scores['threshold']:.2f}) "
      f"| AUC {teacher_scores['auc']:.4f} | {teacher_minutes:.1f} min | peak GPU {teacher_peak_gb:.2f} GB")


Teacher: linear probe on the frozen Phase-1 model (5000 train cases -> dev)
  layer         dev AUC  macro-F1
  embeddings     0.6061    0.5734
  layer 1        0.5914    0.5753
  layer 2        0.5917    0.5814
  layer 3        0.5934    0.5895
  layer 4        0.5858    0.5811
  layer 5        0.5843    0.5758
  layer 6        0.5867    0.5788
  layer 7        0.5860    0.5717
  layer 8        0.5808    0.5733
  layer 9        0.5768    0.5709
  layer 10       0.5787    0.5697
  layer 11       0.5799    0.5678
  layer 12       0.5754    0.5661
  layer 13       0.5766    0.5674
  layer 14       0.5772    0.5670
  layer 15       0.5794    0.5627
  layer 16       0.5739    0.5653
  best frozen layer: embeddings (dev AUC 0.6061)
  feature statistics from 1,423,202 tokens
  Teacher STARTS as a logistic regression on the mixed features: dev AUC 0.6030, macro-F1 0.5794 | 527s

Teacher | train 37387 cases (8000 per epoch) -> dev 1988
Teacher epoch 0 (start = logistic regression) | dev macro-

### CELL 14 · Step 2: teacher probabilities

**In simple words:** **Step 2:** the teacher gives P(accepted) for every train case; the student will learn from these soft answers.

**How it works:** Then the teacher leaves the GPU.

In [14]:
# ============================================================
# CELL 14 — STEP 2: TEACHER PROBABILITIES
#   train cases (the student's training cases) -> case["teacher_prob"] = P(label = 1)
#   dev / test cases -> kept for the report (the student does NOT see them)
#   then the teacher is removed from the GPU
# ============================================================

t0 = time.time()

if TEACHER_SCORED_DOCS is None:
    student_train_cases = train_cases
else:
    student_train_cases = random.Random(SEED + 2).sample(train_cases, min(TEACHER_SCORED_DOCS, len(train_cases)))

for k in range(0, len(student_train_cases), 2000):

    part = student_train_cases[k:k + 2000]
    probs, _ = classify_vectors(teacher, case_vectors(teacher, part))

    for case, p in zip(part, probs):
        case["teacher_prob"] = float(p)

    print(f"  teacher scored {min(k + 2000, len(student_train_cases))}/{len(student_train_cases)} train cases | "
          f"{(time.time() - t0) / 60:.1f} min", flush=True)

t_dev = time.time()
TEACHER_DEV_PROBS, _ = classify_vectors(teacher, case_vectors(teacher, dev_cases))
teacher_dev_seconds = time.time() - t_dev

TEACHER_TEST_PROBS, _ = classify_vectors(teacher, case_vectors(teacher, test_cases))

train_probs = np.array([c["teacher_prob"] for c in student_train_cases])
train_labels = np.array([c["label"] for c in student_train_cases])

print(f"\nTeacher on its train cases: AUC {roc_auc_score(train_labels, train_probs):.4f} | "
      f"probabilities: mean {train_probs.mean():.3f}, 10% {np.percentile(train_probs, 10):.3f}, "
      f"90% {np.percentile(train_probs, 90):.3f}")
print(f"Teacher on dev: AUC {roc_auc_score([c['label'] for c in dev_cases], TEACHER_DEV_PROBS):.4f} | "
      f"dev predicted in {teacher_dev_seconds:.0f}s | step 2 took {(time.time() - t0) / 60:.1f} min")

with open(OUTPUT_DIR / "teacher_test_probabilities.json", "w", encoding="utf-8") as f:
    json.dump({c["id"]: round(float(p), 5) for c, p in zip(test_cases, TEACHER_TEST_PROBS)}, f)

# the teacher is not needed any more: free the GPU for the student
torch.save(trainable_state(teacher), OUTPUT_DIR / "teacher_trainable.pt")
del teacher
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


  teacher scored 2000/37387 train cases | 1.6 min
  teacher scored 4000/37387 train cases | 3.3 min
  teacher scored 6000/37387 train cases | 4.9 min
  teacher scored 8000/37387 train cases | 6.4 min
  teacher scored 10000/37387 train cases | 7.9 min
  teacher scored 12000/37387 train cases | 9.6 min
  teacher scored 14000/37387 train cases | 11.3 min
  teacher scored 16000/37387 train cases | 12.8 min
  teacher scored 18000/37387 train cases | 14.5 min
  teacher scored 20000/37387 train cases | 16.2 min
  teacher scored 22000/37387 train cases | 17.9 min
  teacher scored 24000/37387 train cases | 19.5 min
  teacher scored 26000/37387 train cases | 21.1 min
  teacher scored 28000/37387 train cases | 22.8 min
  teacher scored 30000/37387 train cases | 24.3 min
  teacher scored 32000/37387 train cases | 25.9 min
  teacher scored 34000/37387 train cases | 27.6 min
  teacher scored 36000/37387 train cases | 29.3 min
  teacher scored 37387/37387 train cases | 30.5 min

Teacher on its train 

### CELL 15 · Step 3: the student (Architecture-2 encoder) with knowledge distillation

**In simple words:** **Step 3:** train the student (Architecture-2 encoder) on the gold labels **and** the teacher's probabilities.

**How it works:** loss = (1 − α)·cross-entropy(gold) + α·T²·KL(teacher ‖ student).

In [15]:
# ============================================================
# CELL 15 — STEP 3: THE STUDENT (Architecture-2 encoder) WITH KNOWLEDGE DISTILLATION
#   loss = (1 - KD_ALPHA) * cross-entropy(gold) + KD_ALPHA * T^2 * KL(teacher || student)
# ============================================================

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

model = build_student()                 # `model` = the student (used by the report and the test predictions)
set_trainable(model, TRAIN_EMBEDDINGS)
model.kd_alpha = KD_ALPHA
STUDENT_PROBE = prepare(model, "Student")

print(f"\nStudent | train {len(student_train_cases)} cases with teacher probabilities "
      f"({min(STUDENT_EPOCH_DOCS, len(student_train_cases))} per epoch) -> dev {len(dev_cases)}")

t0 = time.time()
history, best_epoch, _ = train_model(
    model, "Student", STUDENT_WEIGHTS, student_train_cases, STUDENT_EPOCHS, STUDENT_EPOCH_DOCS,
    STUDENT_BATCH_DOCS, STUDENT_GRAD_ACCUM)
train_minutes = (time.time() - t0) / 60

peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

torch.save(trainable_state(model), OUTPUT_DIR / "best_trainable.pt")

print(f"\n✅ Student trained in {train_minutes:.1f} min | best epoch {best_epoch} | peak GPU {peak_gpu_gb:.2f} GB")


Student: linear probe on the frozen Phase-1 model (5000 train cases -> dev)
  layer         dev AUC  macro-F1
  embeddings     0.6000    0.5865
  layer 1        0.5741    0.5634
  layer 2        0.5833    0.5744
  layer 3        0.5907    0.5744
  layer 4        0.5797    0.5811
  layer 5        0.5747    0.5694
  layer 6        0.5749    0.5572
  best frozen layer: embeddings (dev AUC 0.6000)
  feature statistics from 1,423,202 tokens
  Student STARTS as a logistic regression on the mixed features: dev AUC 0.5945, macro-F1 0.5753 | 141s

Student | train 37387 cases with teacher probabilities (12000 per epoch) -> dev 1988
Student epoch 0 (start = logistic regression) | dev macro-F1 @0.35 0.5753 | AUC 0.5945
    batch 500/1500 | loss 0.4102 | 1.3 min
    batch 1000/1500 | loss 0.4183 | 2.3 min
    batch 1500/1500 | loss 0.4227 | 3.2 min
Student epoch 1/4 | loss 0.4227 | dev macro-F1 @0.5 0.6145 | @0.52 0.6157 | AUC 0.6489  <- best | 3.6 min
    layer mix (embeddings, layer 1..): 0.29 0.

### CELL 16 · Dev report (the student) + teacher vs student

**In simple words:** Dev results of the student, and teacher vs student: score, size and prediction time.

In [16]:
# ============================================================
# CELL 16 — DEV REPORT (the STUDENT) + TEACHER vs STUDENT
#   scores + threshold, confusion matrix, errors by document length,
#   which chunks the model uses, false positives / negatives,
#   teacher vs student: macro-F1, size, prediction time
# ============================================================

t0 = time.time()
dev_vectors = case_vectors(model, dev_cases)
dev_probs, dev_attention = classify_vectors(model, dev_vectors)
predict_seconds = time.time() - t0

dev_labels = np.array([c["label"] for c in dev_cases])

THRESHOLD, _ = best_threshold(dev_labels, dev_probs)        # used for the test predictions (CELL 17)
dev_preds = (dev_probs >= THRESHOLD).astype(int)

at_05 = binary_scores(dev_labels, (dev_probs >= 0.5).astype(int))
at_t = binary_scores(dev_labels, dev_preds)
auc = float(roc_auc_score(dev_labels, dev_probs))

print(f"DEV ({RUN_NAME}, best epoch {best_epoch})")
print(f"{'':16s} {'macro-P':>8s} {'macro-R':>8s} {'macro-F1':>9s}")
print(f"{'threshold 0.50':16s} {at_05['precision']:8.4f} {at_05['recall']:8.4f} {at_05['f1']:9.4f}")
print(f"{f'threshold {THRESHOLD:.2f}':16s} {at_t['precision']:8.4f} {at_t['recall']:8.4f} {at_t['f1']:9.4f}")
print(f"ROC-AUC {auc:.4f} | predicted in {predict_seconds:.0f}s")

layer_weights = {n: round(float(w), 4) for n, w in zip(STUDENT_PROBE["layer_names"], model.layer_weights().detach())}
print("\nstudent layer mix after training:", layer_weights)
print(f"student start (logistic regression on the frozen features): dev AUC {STUDENT_PROBE['start']['auc']:.4f}, "
      f"macro-F1 {STUDENT_PROBE['start']['f1']:.4f}")


# ------------------------------------------------------------
# Teacher vs student (only the STUDENT is submitted)
# ------------------------------------------------------------

teacher_threshold, teacher_f1 = best_threshold(dev_labels, TEACHER_DEV_PROBS)
teacher_auc = float(roc_auc_score(dev_labels, TEACHER_DEV_PROBS))
agreement = float(((TEACHER_DEV_PROBS >= teacher_threshold).astype(int) == dev_preds).mean())

kd_comparison = {
    "teacher": {"model": "Architecture-4 hybrid prefix-LM", "backbone_weights": TEACHER_BACKBONE_SIZE,
                "dev_f1": teacher_f1, "dev_auc": teacher_auc, "dev_predict_seconds": teacher_dev_seconds,
                "train_minutes": teacher_minutes, "peak_gpu_gb": teacher_peak_gb, "best_epoch": teacher_best_epoch},
    "student": {"model": "Architecture-2 encoder", "backbone_weights": STUDENT_BACKBONE_SIZE,
                "dev_f1": at_t["f1"], "dev_auc": auc, "dev_predict_seconds": predict_seconds,
                "train_minutes": train_minutes, "peak_gpu_gb": peak_gpu_gb, "best_epoch": best_epoch},
    "dev_agreement": agreement,
}

print(f"\n{'dev':12s} {'weights':>13s} {'macro-F1':>9s} {'AUC':>7s} {'predict dev':>12s}")
for role in ("teacher", "student"):
    r = kd_comparison[role]
    print(f"{role:12s} {r['backbone_weights']:13,d} {r['dev_f1']:9.4f} {r['dev_auc']:7.4f} {r['dev_predict_seconds']:11.0f}s")
print(f"student = {STUDENT_BACKBONE_SIZE / TEACHER_BACKBONE_SIZE:.0%} of the teacher's weights | "
      f"keeps {at_t['f1'] / max(teacher_f1, 1e-9):.0%} of its macro-F1 | they agree on {agreement:.1%} of dev cases")

p, r, f, support = precision_recall_fscore_support(dev_labels, dev_preds, labels=[0, 1], zero_division=0)
print(f"\nper class      P      R      F1   gold")
for k in (0, 1):
    print(f"  label {k}  {p[k]:6.3f} {r[k]:6.3f} {f[k]:6.3f} {support[k]:6d}")

cm = confusion_matrix(dev_labels, dev_preds, labels=[0, 1])
print(f"\nconfusion matrix (rows = gold, columns = predicted)\n          pred 0  pred 1\n  gold 0  {cm[0, 0]:6d}  {cm[0, 1]:6d}\n  gold 1  {cm[1, 0]:6d}  {cm[1, 1]:6d}")


# ------------------------------------------------------------
# Errors by document length (characters), in 4 equal groups
# ------------------------------------------------------------

chars = np.array([len(c["text"]) for c in dev_cases])
edges = np.percentile(chars, [0, 25, 50, 75, 100])

print(f"\n{'length (chars)':>22s} {'cases':>6s} {'macro-F1':>9s} {'false +':>8s} {'false -':>8s}")

length_groups = []

for g in range(4):

    sel = (chars >= edges[g]) & ((chars <= edges[g + 1]) if g == 3 else (chars < edges[g + 1]))

    if sel.sum() == 0:
        continue

    s = binary_scores(dev_labels[sel], dev_preds[sel])
    fp = int(((dev_preds == 1) & (dev_labels == 0) & sel).sum())
    fn = int(((dev_preds == 0) & (dev_labels == 1) & sel).sum())

    length_groups.append({"min_chars": int(edges[g]), "max_chars": int(edges[g + 1]), "cases": int(sel.sum()),
                          "f1": s["f1"], "false_pos": fp, "false_neg": fn})

    print(f"{int(edges[g]):>10d} - {int(edges[g + 1]):>9d} {int(sel.sum()):6d} {s['f1']:9.4f} {fp:8d} {fn:8d}")


# ------------------------------------------------------------
# Which chunks matter?
#   1. average attention weight per chunk position (counted from the end
#      for DOC_PART "end", from the start otherwise)
#   2. macro-F1 when the classifier sees ONLY one chunk position
# ------------------------------------------------------------

from_end = DOC_PART == "end"

attention_by_position = {}
for w in dev_attention:
    for k, value in enumerate(w[::-1] if from_end else w):
        attention_by_position.setdefault(k, []).append(float(value))

single_chunk = {}

for k in range(max(len(v) for v in dev_vectors)):

    subset, idx = [], []

    for i, v in enumerate(dev_vectors):
        if len(v) > k:
            subset.append(v[len(v) - 1 - k:len(v) - k] if from_end else v[k:k + 1])
            idx.append(i)

    if not subset:
        continue

    probs_k, _ = classify_vectors(model, subset)
    single_chunk[k] = {"cases": len(idx), "f1": binary_scores(dev_labels[idx], (probs_k >= THRESHOLD).astype(int))["f1"]}

name = "chunk (0 = last)" if from_end else "chunk (0 = first)"

if DOC_PART == "start_end":
    print("(start_end: the first chunks come from the start of the document, the last ones from its end)")
print(f"\n{name:18s} {'mean attention':>15s} {'F1 with only this chunk':>24s}")

for k in sorted(attention_by_position):
    print(f"{k:18d} {np.mean(attention_by_position[k]):15.3f} {single_chunk.get(k, {}).get('f1', 0):24.4f}")

print(f"{'all chunks':18s} {'':15s} {at_t['f1']:24.4f}")


# ------------------------------------------------------------
# False positives / negatives: saved for reading
# ------------------------------------------------------------

errors = []

for case, label, pred, prob in zip(dev_cases, dev_labels, dev_preds, dev_probs):
    if label != pred:
        errors.append({"id": case["id"], "gold": int(label), "pred": int(pred), "prob_1": round(float(prob), 4),
                       "type": "false positive" if pred == 1 else "false negative",
                       "chars": len(case["text"]), "end_of_text": case["text"][-600:]})

fp_chars = [e["chars"] for e in errors if e["type"] == "false positive"]
fn_chars = [e["chars"] for e in errors if e["type"] == "false negative"]
ok_chars = [len(c["text"]) for c, l, p in zip(dev_cases, dev_labels, dev_preds) if l == p]

print(f"\nmedian length (chars): correct {np.median(ok_chars):.0f} | "
      f"false positives {np.median(fp_chars) if fp_chars else 0:.0f} ({len(fp_chars)}) | "
      f"false negatives {np.median(fn_chars) if fn_chars else 0:.0f} ({len(fn_chars)})")

with open(OUTPUT_DIR / "dev_errors.json", "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=1)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

results = {
    "run": RUN_NAME,
    "model": "STUDENT Architecture-2 encoder distilled from TEACHER Architecture-4 (both: head + token embeddings trained, mix of all layers + linear path + token attention / chunk BiLSTM)",
    "knowledge_distillation": {"alpha": KD_ALPHA, "temperature": KD_TEMPERATURE,
                               "student_train_cases": len(student_train_cases), **kd_comparison},
    "trained_on": "train only (dev used to pick the best epoch and the threshold)",
    "doc_part": DOC_PART, "doc_tokens": DOC_TOKENS, "chunk_len": CHUNK_LEN,
    "fine_tuning": {"trained": "head + token embeddings" if TRAIN_EMBEDDINGS else "head only",
                    "lr_embeddings": LR_EMBEDDINGS, "lr_head": LR_HEAD},
    "best_epoch": best_epoch, "train_minutes": train_minutes, "peak_gpu_gb": peak_gpu_gb,
    "dev_predict_seconds": predict_seconds,
    "threshold": THRESHOLD,
    "layer_mix": layer_weights,
    "start_logistic_regression": {"teacher": TEACHER_PROBE["start"], "student": STUDENT_PROBE["start"]},
    "probe_frozen_features": {"teacher": TEACHER_PROBE, "student": STUDENT_PROBE},
    "teacher_history": teacher_history,
    "dev_at_0.5": at_05, "dev_at_threshold": at_t, "dev_auc": auc,
    "dev_confusion_matrix": cm.tolist(),
    "dev_by_length": length_groups,
    "attention_by_chunk": {k: float(np.mean(v)) for k, v in attention_by_position.items()},
    "f1_single_chunk": single_chunk,
    "history": history,
}

with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with open(OUTPUT_DIR / "dev_probabilities.json", "w", encoding="utf-8") as f:
    json.dump({c["id"]: round(float(p), 5) for c, p in zip(dev_cases, dev_probs)}, f)

print("\nSaved:", OUTPUT_DIR / "results.json", ", dev_errors.json, dev_probabilities.json")

DEV (p4_kd_arch4_to_arch2_v3, best epoch 3)
                  macro-P  macro-R  macro-F1
threshold 0.50     0.6254   0.5392    0.4437
threshold 0.22     0.6404   0.6398    0.6395
ROC-AUC 0.6711 | predicted in 24s

student layer mix after training: {'embeddings': 0.269, 'layer 1': 0.088, 'layer 2': 0.1384, 'layer 3': 0.2003, 'layer 4': 0.1172, 'layer 5': 0.0927, 'layer 6': 0.0944}
student start (logistic regression on the frozen features): dev AUC 0.5945, macro-F1 0.5753

dev                weights  macro-F1     AUC  predict dev
teacher        205,496,064    0.6304  0.6568         102s
student         67,894,272    0.6395  0.6711          24s

per class      P      R      F1   gold
  label 0   0.632  0.670  0.650    994
  label 1   0.649  0.610  0.629    994

confusion matrix (rows = gold, columns = predicted)
          pred 0  pred 1
  gold 0     666     328
  gold 1     388     606

        length (chars)  cases  macro-F1  false +  false -
       904 -     10667    496    0.6812      

### CELL 17 · Predict test with the student + write prediction.json + submission.zip

**In simple words:** Predict every test case and write `submission.zip`.

**How it works:** `prediction.json` = `[{"id": "case_001", "pred": 1}, …]`.

In [17]:
# ============================================================
# CELL 17 — PREDICT TEST WITH THE STUDENT + WRITE prediction.json + submission.zip
#   format: [{"id": "case_001", "pred": 1}, ...]
# ============================================================

t0 = time.time()
test_probs, _ = classify_vectors(model, case_vectors(model, test_cases))
print(f"Predicted {len(test_cases)} test cases in {time.time() - t0:.0f}s (threshold {THRESHOLD:.2f})")

submission = [{"id": case["id"], "pred": int(prob >= THRESHOLD)} for case, prob in zip(test_cases, test_probs)]


# ------------------------------------------------------------
# Format checks
# ------------------------------------------------------------

assert len(submission) == len(test_cases)
assert [s["id"] for s in submission] == [c["id"] for c in test_cases]
assert all(isinstance(s["pred"], int) and s["pred"] in (0, 1) for s in submission)

counts = Counter(s["pred"] for s in submission)
print(f"✅ {len(submission)} predictions | pred 0: {counts[0]} | pred 1: {counts[1]}")


PREDICTION_FILE = OUTPUT_DIR / "prediction.json"
ZIP_FILE = OUTPUT_DIR / "submission.zip"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=1)

with open(OUTPUT_DIR / "test_probabilities.json", "w", encoding="utf-8") as f:
    json.dump({c["id"]: round(float(p), 5) for c, p in zip(test_cases, test_probs)}, f)

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")        # file at the ROOT of the zip

with zipfile.ZipFile(ZIP_FILE) as zf:
    print("submission.zip:", zf.namelist())

print("Saved:", ZIP_FILE)

Predicted 1517 test cases in 18s (threshold 0.22)
✅ 1517 predictions | pred 0: 801 | pred 1: 716
submission.zip: ['prediction.json']
Saved: /kaggle/working/p4_kd_arch4_to_arch2_v3/submission.zip
